# 3 実装の直接比較 — 精度だけを見ない

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 02 まず自動化しない——個別モデルの直接比較

**このノートのパス**　`ml-code/05_hpo/01_model_benchmark.ipynb`

**このノートで確かめること**
XGBoost / LightGBM / CatBoost を**固定の設定・同一条件**で並べ、
精度・学習時間・推論時間・メモリを**同じ 1 回のループで**測る。
「どれが最良か」は指標を 1 つ決めない限り答えが出ないことを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `xgboost lightgbm catboost scikit-learn psutil numpy` |
| 実行時間の目安 | 約 2〜5 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q xgboost lightgbm catboost scikit-learn psutil numpy

In [3]:
import time, os
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

SEED = 42

# 陽性 5% の不均衡データ。実データの代わりに、まず合成データで手順を固める
X, y = make_classification(
    n_samples=300_000, n_features=30, n_informative=15, n_redundant=8,
    weights=[0.95, 0.05], class_sep=1.3, random_state=SEED,
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)   # stratify は不均衡では必須

# 不均衡の補正倍率。3 実装で同じ値を使い、条件を揃える
pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print("train", X_tr.shape, "pos_weight", round(pos_weight, 2))

import psutil
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

train (240000, 30) pos_weight 17.39


In [4]:
models = {
    "XGBoost":  XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=pos_weight, random_state=SEED, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                               class_weight="balanced", random_state=SEED,
                               n_jobs=-1, verbosity=-1),
    "CatBoost": CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
                                   scale_pos_weight=pos_weight,
                                   random_seed=SEED, verbose=0),
}

proc = psutil.Process(os.getpid())
rows = []
for name, model in models.items():
    mem0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter(); model.fit(X_tr, y_tr); fit_sec = time.perf_counter() - t0
    mem1 = proc.memory_info().rss / 1024**2

    t0 = time.perf_counter(); prob = model.predict_proba(X_te)[:, 1]
    pred_sec = time.perf_counter() - t0
    pred = (prob >= 0.5).astype(int)

    rows.append(dict(Model=name,
        ROC_AUC=roc_auc_score(y_te, prob), PR_AUC=average_precision_score(y_te, prob),
        Precision=precision_score(y_te, pred), Recall=recall_score(y_te, pred),
        F1=f1_score(y_te, pred), fit_sec=fit_sec, pred_sec=pred_sec,
        mem_MB=mem1 - mem0))
    print(f"{name:9s} PR-AUC={rows[-1]['PR_AUC']:.4f} fit={fit_sec:.1f}s "
          f"pred={pred_sec:.2f}s mem+={rows[-1]['mem_MB']:.0f}MB", flush=True)

import pandas as pd
print(pd.DataFrame(rows).round(4).to_string(index=False))

XGBoost   PR-AUC=0.9038 fit=2.1s pred=0.02s mem+=88MB


LightGBM  PR-AUC=0.8999 fit=5.4s pred=0.08s mem+=125MB


CatBoost  PR-AUC=0.9048 fit=3.8s pred=0.01s mem+=123MB


   Model  ROC_AUC  PR_AUC  Precision  Recall     F1  fit_sec  pred_sec   mem_MB
 XGBoost   0.9538  0.9038     0.7555  0.8918 0.8180   2.1205    0.0159  88.0000
LightGBM   0.9534  0.8999     0.7084  0.8912 0.7894   5.4037    0.0829 124.6094
CatBoost   0.9546  0.9048     0.7015  0.8967 0.7872   3.7814    0.0066 123.3125


## 読み取りどころ

読み取るべきことは **「XGBoost が勝った」ではない**。

- **PR-AUC の差はごく小さいのに、学習時間は数倍違う。**日に何度も再学習する系なら、わずかな精度を捨てて時間を取るほうが正しい。
- **推論時間の順位は学習時間の順位と一致しない。**リアルタイム推論なら評価が変わる。
- **メモリはコンテナのサイズと同時実行数を決める。**

⚠️ **測っていない量は、議論の場に出せない。**精度だけの表は選定の材料として半分しかない。

⚠️ 秒数は環境で大きく変わる。**比率と桁を読み、必ず自分の環境で測り直すこと。**